# Silver Layer

### Objective

Transform the Bronze data into a clean and standardized dataset by applying all business rules.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable
import re

CATALOG = "workspace"
SCHEMA = "hrm6131_landing"

# Bronze Tables
BRONZE_PROFILES = f"{CATALOG}.{SCHEMA}.bronze_profiles"
BRONZE_DAY1 = f"{CATALOG}.{SCHEMA}.bronze_day1"
BRONZE_DAY2 = f"{CATALOG}.{SCHEMA}.bronze_day2"

# Silver Table
SILVER_TICKETS = f"{CATALOG}.{SCHEMA}.silver_tickets"

print("Configuration loaded.")

Configuration loaded.


## Read Bronze Tables

In [0]:
bronze_profiles = spark.table(BRONZE_PROFILES)
bronze_day1 = spark.table(BRONZE_DAY1)
bronze_day2 = spark.table(BRONZE_DAY2)

In [0]:
print("Profiles :", bronze_profiles.count())
print("Day 1 :", bronze_day1.count())
print("Day 2 :", bronze_day2.count())

Profiles : 42
Day 1 : 123
Day 2 : 86


## Merge Day 1 and Day 2 Tickets

In [0]:
print("Bronze Day1 Columns")
print(bronze_day1.columns)

print("\nBronze Day2 Columns")
print(bronze_day2.columns)

Bronze Day1 Columns
['ticket_id', 'agent_id', 'status', 'resolution_time', 'ticket_category', 'day', 'ingestion_timestamp', 'source_file']

Bronze Day2 Columns
['ticket_id', 'agent_id', 'status', 'resolution_time', 'ticket_category', 'day', 'ingestion_timestamp', 'source_file']


In [0]:
bronze_day1.printSchema()
bronze_day2.printSchema()

root
 |-- ticket_id: string (nullable = true)
 |-- agent_id: string (nullable = true)
 |-- status: string (nullable = true)
 |-- resolution_time: string (nullable = true)
 |-- ticket_category: string (nullable = true)
 |-- day: integer (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)

root
 |-- ticket_id: string (nullable = true)
 |-- agent_id: string (nullable = true)
 |-- status: string (nullable = true)
 |-- resolution_time: string (nullable = true)
 |-- ticket_category: string (nullable = true)
 |-- day: integer (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



In [0]:
tickets = bronze_day1.unionByName(bronze_day2)

print("Total Tickets :", tickets.count())

display(tickets)

Total Tickets : 209


ticket_id,agent_id,status,resolution_time,ticket_category,day,ingestion_timestamp,source_file
TKT00001,A001,Resolved,0h 35m 45s,Technical,1,2026-07-18T19:38:03.498Z,day1_tickets.csv
TKT00002,A001,Resolved,0h 22m 10s,Billing,1,2026-07-18T19:38:03.498Z,day1_tickets.csv
TKT00003,A001,Resolved,0h 12m 5s,General,1,2026-07-18T19:38:03.498Z,day1_tickets.csv
TKT00004,A001,Open,0h 40m 0s,Delivery,1,2026-07-18T19:38:03.498Z,day1_tickets.csv
TKT00005,A002,Resolved,1h 10m 30s,Returns,1,2026-07-18T19:38:03.498Z,day1_tickets.csv
TKT00006,A002,Resolved,0h 16m 26s,Billing,1,2026-07-18T19:38:03.498Z,day1_tickets.csv
TKT00007,A002,Pending,0h 36m 0s,Returns,1,2026-07-18T19:38:03.498Z,day1_tickets.csv
TKT00008,A003,Resolved,0h 25m 50s,Technical,1,2026-07-18T19:38:03.498Z,day1_tickets.csv
TKT00009,A003,Resolved,0h 15m 0s,Billing,1,2026-07-18T19:38:03.498Z,day1_tickets.csv
TKT00010,A003,Resolved,0h 18m 44s,Account,1,2026-07-18T19:38:03.498Z,day1_tickets.csv


## Remove Duplicate Records

In [0]:
tickets_clean = tickets.dropDuplicates(["ticket_id"])

print("Tickets before removing duplicates :", tickets.count())
print("Tickets after removing duplicates  :", tickets_clean.count())

Tickets before removing duplicates : 209
Tickets after removing duplicates  : 208


## Join Agent Profiles

In [0]:
silver_tickets = tickets_clean.join(
    bronze_profiles.select(
        "agent_id",
        "agent_name",
        "role",
        "team_lead_id"
    ),
    on="agent_id",
    how="left"
)

display(silver_tickets)

agent_id,ticket_id,status,resolution_time,ticket_category,day,ingestion_timestamp,source_file,agent_name,role,team_lead_id
A001,TKT00001,Resolved,0h 35m 45s,Technical,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A001,Junior Support Agent,TL01
A001,TKT00002,Resolved,0h 22m 10s,Billing,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A001,Junior Support Agent,TL01
A001,TKT00003,Resolved,0h 12m 5s,General,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A001,Junior Support Agent,TL01
A017,TKT00050,Pending,0h 33m 0s,Returns,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A017,Support Agent,TL04
A022,TKT00061,Resolved,0h 44m 0s,Account,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A022,Support Agent,TL05
A025,TKT00069,Resolved,0h 26m 0s,Delivery,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A025,Senior Support Agent,TL05
A027,TKT00077,Resolved,0h 11m 0s,Technical,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A027,Junior Support Agent,TL06
A030,TKT00084,Open,0h 40m 0s,Delivery,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A030,Junior Support Agent,TL06
A033,TKT00094,Resolved,0h 19m 45s,Account,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A033,Junior Support Agent,TL07
A004,TKT00011,Resolved,0h 33m 15s,Delivery,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A004,Junior Support Agent,TL01


## Filter Team Scope

In [0]:
silver_tickets = silver_tickets.filter(
    F.col("team_lead_id").isin(
        "TL01",
        "TL02",
        "TL03",
        "TL04",
        "TL05",
        "TL06",
        "TL07",
        "TL08"
    )
)

print("Rows after team filter :", silver_tickets.count())

display(silver_tickets)

Rows after team filter : 197


agent_id,ticket_id,status,resolution_time,ticket_category,day,ingestion_timestamp,source_file,agent_name,role,team_lead_id
A001,TKT00001,Resolved,0h 35m 45s,Technical,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A001,Junior Support Agent,TL01
A001,TKT00002,Resolved,0h 22m 10s,Billing,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A001,Junior Support Agent,TL01
A001,TKT00003,Resolved,0h 12m 5s,General,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A001,Junior Support Agent,TL01
A017,TKT00050,Pending,0h 33m 0s,Returns,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A017,Support Agent,TL04
A022,TKT00061,Resolved,0h 44m 0s,Account,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A022,Support Agent,TL05
A025,TKT00069,Resolved,0h 26m 0s,Delivery,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A025,Senior Support Agent,TL05
A027,TKT00077,Resolved,0h 11m 0s,Technical,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A027,Junior Support Agent,TL06
A030,TKT00084,Open,0h 40m 0s,Delivery,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A030,Junior Support Agent,TL06
A033,TKT00094,Resolved,0h 19m 45s,Account,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A033,Junior Support Agent,TL07
A004,TKT00011,Resolved,0h 33m 15s,Delivery,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A004,Junior Support Agent,TL01


## Remove Invalid Resolution Times

In [0]:
silver_tickets = silver_tickets.filter(
    F.col("resolution_time").rlike(r"^\d+h \d+m \d+s$")
)

print("Rows after removing invalid time records:")
print(silver_tickets.count())

Rows after removing invalid time records:
194


In [0]:
silver_tickets.select("resolution_time").distinct().display()

resolution_time
0h 35m 45s
0h 22m 10s
0h 12m 5s
0h 33m 0s
0h 44m 0s
0h 26m 0s
0h 11m 0s
0h 40m 0s
0h 19m 45s
0h 33m 15s


## Convert Resolution Time

In [0]:
silver_tickets = (
    silver_tickets
    .withColumn(
        "hours",
        F.expr("try_cast(regexp_extract(resolution_time, '(\\\\d+)h', 1) as int)")
    )
    .withColumn(
        "minutes",
        F.expr("try_cast(regexp_extract(resolution_time, '(\\\\d+)m', 1) as int)")
    )
    .withColumn(
        "seconds",
        F.expr("try_cast(regexp_extract(resolution_time, '(\\\\d+)s', 1) as int)")
    )
)

In [0]:
silver_tickets = silver_tickets.withColumn(
    "resolution_time_minutes",
    (
        F.regexp_extract("resolution_time", r"(\d+)h", 1).cast("int") * 60
        + F.regexp_extract("resolution_time", r"(\d+)m", 1).cast("int")
        + F.when(
            F.regexp_extract("resolution_time", r"(\d+)s", 1).cast("int") >= 30,
            1
        ).otherwise(0)
    )
)

In [0]:
#Resolution time conversion
display(
    silver_tickets.select(
        "resolution_time",
        "resolution_time_minutes"
    )
)

resolution_time,resolution_time_minutes
0h 35m 45s,36
0h 22m 10s,22
0h 12m 5s,12
0h 33m 0s,33
0h 44m 0s,44
0h 26m 0s,26
0h 11m 0s,11
0h 40m 0s,40
0h 19m 45s,20
0h 33m 15s,33


## Filter Resolved Tickets

In [0]:
silver_tickets = silver_tickets.filter(
    F.col("status") == "Resolved"
)

print("Resolved Tickets:", silver_tickets.count())

Resolved Tickets: 152


## Apply Quality Threshold

In [0]:
silver_tickets = silver_tickets.filter(
    F.col("resolution_time_minutes") > 15
)

print("Tickets after quality threshold:", silver_tickets.count())

display(silver_tickets)

Tickets after quality threshold: 134


agent_id,ticket_id,status,resolution_time,ticket_category,day,ingestion_timestamp,source_file,agent_name,role,team_lead_id,hours,minutes,seconds,resolution_time_minutes
A001,TKT00001,Resolved,0h 35m 45s,Technical,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A001,Junior Support Agent,TL01,0,35,45,36
A001,TKT00002,Resolved,0h 22m 10s,Billing,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A001,Junior Support Agent,TL01,0,22,10,22
A022,TKT00061,Resolved,0h 44m 0s,Account,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A022,Support Agent,TL05,0,44,0,44
A025,TKT00069,Resolved,0h 26m 0s,Delivery,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A025,Senior Support Agent,TL05,0,26,0,26
A033,TKT00094,Resolved,0h 19m 45s,Account,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A033,Junior Support Agent,TL07,0,19,45,20
A004,TKT00011,Resolved,0h 33m 15s,Delivery,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A004,Junior Support Agent,TL01,0,33,15,33
A016,TKT00045,Resolved,0h 36m 0s,General,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A016,Junior Support Agent,TL04,0,36,0,36
A026,TKT00072,Resolved,0h 23m 0s,Billing,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A026,Support Agent,TL06,0,23,0,23
A031,TKT00086,Resolved,0h 29m 0s,Billing,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A031,Senior Support Agent,TL07,0,29,0,29
A034,TKT00096,Resolved,0h 25m 15s,Returns,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A034,Support Agent,TL07,0,25,15,25


## Apply Day 2 Carry-over Rule

In [0]:
# Successful Day 1 agents
day1_success_agents = (
    silver_tickets
    .filter(F.col("day") == 1)
    .select("agent_id")
    .distinct()
)
# Remove Day 2 records of successful Day 1 agents
silver_tickets = (
    silver_tickets
    .filter(
        (F.col("day") == 1) |
        (
            (F.col("day") == 2) &
            (~F.col("agent_id").isin(
                [row.agent_id for row in day1_success_agents.collect()]
            ))
        )
    )
)

print(
    "After Day2 carry-over rule:",
    silver_tickets.count()
)

display(silver_tickets)

After Day2 carry-over rule: 77


agent_id,ticket_id,status,resolution_time,ticket_category,day,ingestion_timestamp,source_file,agent_name,role,team_lead_id,hours,minutes,seconds,resolution_time_minutes
A001,TKT00001,Resolved,0h 35m 45s,Technical,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A001,Junior Support Agent,TL01,0,35,45,36
A001,TKT00002,Resolved,0h 22m 10s,Billing,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A001,Junior Support Agent,TL01,0,22,10,22
A022,TKT00061,Resolved,0h 44m 0s,Account,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A022,Support Agent,TL05,0,44,0,44
A025,TKT00069,Resolved,0h 26m 0s,Delivery,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A025,Senior Support Agent,TL05,0,26,0,26
A033,TKT00094,Resolved,0h 19m 45s,Account,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A033,Junior Support Agent,TL07,0,19,45,20
A004,TKT00011,Resolved,0h 33m 15s,Delivery,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A004,Junior Support Agent,TL01,0,33,15,33
A016,TKT00045,Resolved,0h 36m 0s,General,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A016,Junior Support Agent,TL04,0,36,0,36
A026,TKT00072,Resolved,0h 23m 0s,Billing,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A026,Support Agent,TL06,0,23,0,23
A031,TKT00086,Resolved,0h 29m 0s,Billing,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A031,Senior Support Agent,TL07,0,29,0,29
A034,TKT00096,Resolved,0h 25m 15s,Returns,1,2026-07-18T19:38:03.498Z,day1_tickets.csv,Agent_A034,Support Agent,TL07,0,25,15,25


In [0]:
# Validation
print("Null validation")

silver_tickets.select(
    [
        F.count(
            F.when(F.col(c).isNull(), c)
        ).alias(c)
        for c in silver_tickets.columns
    ]
).show()

Null validation
+--------+---------+------+---------------+---------------+---+-------------------+-----------+----------+----+------------+-----+-------+-------+-----------------------+
|agent_id|ticket_id|status|resolution_time|ticket_category|day|ingestion_timestamp|source_file|agent_name|role|team_lead_id|hours|minutes|seconds|resolution_time_minutes|
+--------+---------+------+---------------+---------------+---+-------------------+-----------+----------+----+------------+-----+-------+-------+-----------------------+
|       0|        0|     0|              0|              0|  0|                  0|          0|         0|   0|           0|    0|      0|      0|                      0|
+--------+---------+------+---------------+---------------+---+-------------------+-----------+----------+----+------------+-----+-------+-------+-----------------------+



In [0]:
spark.sql("SHOW TABLES IN workspace.hrm6131_landing").show()

+---------------+--------------------+-----------+
|       database|           tableName|isTemporary|
+---------------+--------------------+-----------+
|hrm6131_landing|      agent_profiles|      false|
|hrm6131_landing|         bronze_day1|      false|
|hrm6131_landing|         bronze_day2|      false|
|hrm6131_landing|     bronze_profiles|      false|
|hrm6131_landing|        day1_tickets|      false|
|hrm6131_landing|        day2_tickets|      false|
|hrm6131_landing|gold_agent_perfor...|      false|
|hrm6131_landing| gold_day2_carryover|      false|
|hrm6131_landing|gold_quality_comp...|      false|
|hrm6131_landing|gold_team_perform...|      false|
|hrm6131_landing|      silver_tickets|      false|
+---------------+--------------------+-----------+



## Save Silver Table

In [0]:
(
    silver_tickets.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TICKETS)
)

print("Silver table created successfully.")

Silver table created successfully.


In [0]:
silver_tickets.printSchema()

root
 |-- agent_id: string (nullable = true)
 |-- ticket_id: string (nullable = true)
 |-- status: string (nullable = true)
 |-- resolution_time: string (nullable = true)
 |-- ticket_category: string (nullable = true)
 |-- day: integer (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- agent_name: string (nullable = true)
 |-- role: string (nullable = true)
 |-- team_lead_id: string (nullable = true)
 |-- hours: integer (nullable = true)
 |-- minutes: integer (nullable = true)
 |-- seconds: integer (nullable = true)
 |-- resolution_time_minutes: integer (nullable = true)



In [0]:
display(
    silver_tickets.select(
        "ticket_id",
        "agent_id",
        "agent_name",
        "team_lead_id",
        "resolution_time_minutes"
    )
)

ticket_id,agent_id,agent_name,team_lead_id,resolution_time_minutes
TKT00001,A001,Agent_A001,TL01,36
TKT00002,A001,Agent_A001,TL01,22
TKT00061,A022,Agent_A022,TL05,44
TKT00069,A025,Agent_A025,TL05,26
TKT00094,A033,Agent_A033,TL07,20
TKT00011,A004,Agent_A004,TL01,33
TKT00045,A016,Agent_A016,TL04,36
TKT00072,A026,Agent_A026,TL06,23
TKT00086,A031,Agent_A031,TL07,29
TKT00096,A034,Agent_A034,TL07,25
